# 1. Pré-processamento

Lê cada arquivo da pasta `data/`, detecta a coluna de data (1ª; se não, 2ª),
mantém a última coluna como target, salva `[date, y]` em `data_processed/`
e imprime estatísticas e diagnóstico.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")

DATA_DIR = "data"
OUT_DIR  = "data_processed"

In [ ]:
# =========================================================
# 1. DETECÇÃO E LIMPEZA
# =========================================================

def is_date_column(col):
    """Retorna (True, série_convertida) se >=80% dos valores parecem datas."""
    try:
        conv = pd.to_datetime(col, errors="coerce")
    except Exception:
        return False, None
    if conv.notna().sum() < len(col) * 0.8:
        return False, None
    years = conv.dt.year.dropna()
    if years.empty or years.min() < 1900 or years.max() > 2100:
        return False, None
    return True, conv


def detect_date_column(df):
    """Tenta primeiro a coluna 0; se falhar, tenta a coluna 1."""
    for idx in [0, 1]:
        if idx >= df.shape[1]:
            continue
        ok, conv = is_date_column(df.iloc[:, idx])
        if ok:
            return idx, conv
    return None, None


def preprocess_to_univariate(filepath):
    """Lê o arquivo e devolve DataFrame [date, y] limpo e ordenado."""
    if filepath.lower().endswith(".csv"):
        df = pd.read_csv(filepath)
    else:
        df = pd.read_excel(filepath)

    date_idx, dates = detect_date_column(df)
    if date_idx is None:
        raise ValueError("nenhuma das duas primeiras colunas contém datas")

    target = pd.to_numeric(df.iloc[:, -1], errors="coerce")
    out = (pd.DataFrame({"date": dates, "y": target.values})
             .dropna()
             .sort_values("date")
             .reset_index(drop=True))
    return out

In [ ]:
# =========================================================
# 2. ESTATÍSTICAS E DIAGNÓSTICOS
# =========================================================

def detect_frequency(dates):
    if len(dates) < 2:
        return "desconhecida"
    days = dates.diff().dt.total_seconds().dropna().median() / 86400
    if days <= 1.5:  return "diária"
    if days <= 8:    return "semanal"
    if days <= 16:   return "quinzenal"
    if days <= 35:   return "mensal"
    if days <= 100:  return "trimestral"
    if days <= 200:  return "semestral"
    return "anual"


def stationarity_tests(y):
    """ADF + KPSS combinados.  Retorna p-valores e conclusão qualitativa."""
    out = {"ADF_p": None, "KPSS_p": None, "conclusao": "inconclusiva"}
    try:
        out["ADF_p"]  = float(adfuller(y, autolag="AIC")[1])  # H0: tem raiz unitária
    except Exception:
        pass
    try:
        out["KPSS_p"] = float(kpss(y, regression="c", nlags="auto")[1])  # H0: estacionária
    except Exception:
        pass

    adf_est  = out["ADF_p"]  is not None and out["ADF_p"]  < 0.05  # rejeita raiz unitária
    kpss_est = out["KPSS_p"] is not None and out["KPSS_p"] > 0.05  # não rejeita estacionariedade

    if   adf_est and kpss_est:         out["conclusao"] = "estacionária"
    elif (not adf_est) and (not kpss_est): out["conclusao"] = "não estacionária"
    elif adf_est and (not kpss_est):   out["conclusao"] = "estacionária por diferenças"
    elif (not adf_est) and kpss_est:   out["conclusao"] = "tendência determinística"
    return out


def detect_best_period(y, candidates=(7, 12, 24, 30, 52)):
    """Escolhe entre candidatos sazonais o de maior |ACF|."""
    max_lag = min(len(y) // 2, 60)
    if max_lag < 4:
        return None
    try:
        ac = acf(y, nlags=max_lag, fft=True)
    except Exception:
        return None
    viable = [p for p in candidates if p < max_lag]
    if not viable:
        return None
    best = max(viable, key=lambda p: abs(ac[p]))
    return best, float(ac[best])


def stl_strengths(y, period):
    """
    Força sazonal F_S e força de tendência F_T, ambas em [0,1]:
        F_T = max(0, 1 - Var(R) / Var(T+R))
        F_S = max(0, 1 - Var(R) / Var(S+R))
    Retorna também a razão de ruído std(R)/std(y).
    """
    if len(y) < 2 * period:
        return None
    try:
        stl = STL(y, period=period, robust=True).fit()
        T, S, R = stl.trend, stl.seasonal, stl.resid
        var_R = np.var(R)
        F_T = max(0.0, 1 - var_R / max(np.var(T + R), 1e-12))
        F_S = max(0.0, 1 - var_R / max(np.var(S + R), 1e-12))
        noise = np.std(R) / (np.std(y) + 1e-12)
        return float(F_T), float(F_S), float(noise)
    except Exception:
        return None


def classify(v, low, high, labels):
    """Classifica em 3 níveis dado limiares low/high."""
    if v < low:  return labels[0]
    if v < high: return labels[1]
    return labels[2]


def describe_and_diagnose(df, name):
    y = df["y"].values
    n = len(y)
    print(f"\n{'=' * 70}")
    print(f"  DATASET: {name}")
    print('=' * 70)

    # --- dimensão ---
    print(f"\n[DIMENSÃO]")
    print(f"  Observações:        {n}")
    print(f"  Período:            {df['date'].min().date()} → {df['date'].max().date()}")
    print(f"  Frequência:         {detect_frequency(df['date'])}")

    # --- descritivas ---
    print(f"\n[DESCRITIVAS]")
    print(f"  Média / Mediana:    {np.mean(y):.4f} / {np.median(y):.4f}")
    print(f"  Desvio padrão:      {np.std(y):.4f}")
    cv = np.std(y) / abs(np.mean(y)) if np.mean(y) != 0 else np.nan
    print(f"  Coef. de variação:  {cv:.4f}")
    print(f"  Mín / Máx:          {np.min(y):.4f} / {np.max(y):.4f}")
    print(f"  Assimetria:         {scipy_stats.skew(y):.4f}")
    print(f"  Curtose excesso:    {scipy_stats.kurtosis(y):.4f}")
    print(f"  Tem zeros/neg:      {'sim' if (y <= 0).any() else 'não'}")

    # --- estacionariedade ---
    sty = stationarity_tests(y)
    print(f"\n[ESTACIONARIEDADE]")
    print(f"  ADF  p-valor:       {sty['ADF_p']}  (H0: tem raiz unitária)")
    print(f"  KPSS p-valor:       {sty['KPSS_p']}  (H0: estacionária)")
    print(f"  Conclusão:          {sty['conclusao']}")

    # --- sazonalidade ---
    print(f"\n[SAZONALIDADE / TENDÊNCIA / RUÍDO]")
    period_info = detect_best_period(y)
    F_T = F_S = noise = None
    if period_info is None:
        print(f"  Não foi possível avaliar (série curta demais).")
    else:
        best_p, ac_p = period_info
        print(f"  Período candidato:  {best_p}  (ACF nesse lag = {ac_p:.4f})")
        s = stl_strengths(y, best_p)
        if s is not None:
            F_T, F_S, noise = s
            print(f"  Força sazonal:      {F_S:.4f}  ({classify(F_S, 0.3, 0.6, ['fraca', 'moderada', 'forte'])})")
            print(f"  Força tendência:    {F_T:.4f}  ({classify(F_T, 0.3, 0.6, ['fraca', 'moderada', 'forte'])})")
            print(f"  Razão de ruído:     {noise:.4f}  ({classify(noise, 0.3, 0.6, ['baixo', 'moderado', 'alto'])})")

    # --- outliers ---
    q1, q3 = np.percentile(y, [25, 75])
    iqr = q3 - q1
    n_out = int(((y < q1 - 1.5 * iqr) | (y > q3 + 1.5 * iqr)).sum())
    print(f"\n[OUTLIERS]")
    print(f"  Pela regra IQR:     {n_out} ({n_out / n * 100:.1f}%)")

    # --- diagnóstico final ---
    print(f"\n[DIAGNÓSTICO]")
    notes = []
    if "não estacionária" in sty["conclusao"]:
        notes.append("Não estacionária → diferenciação ajuda (Grupo B ou D).")
    elif sty["conclusao"] == "estacionária":
        notes.append("Estacionária → modelos lineares e ARIMA simples competem bem.")

    if F_S is not None:
        if F_S > 0.6:
            notes.append(f"Sazonalidade forte (período ~{period_info[0]}) → diferenciação sazonal recomendada.")
        elif F_S > 0.3:
            notes.append(f"Sazonalidade moderada (período ~{period_info[0]}).")
        if F_T is not None and F_T > 0.6:
            notes.append("Tendência forte → modelos com termo de tendência ou diferenciação ajudam.")
        if noise is not None and noise > 0.5:
            notes.append("Bastante ruído → regularização (weight_decay, dropout, max_depth menor) tende a ajudar.")

    sk = scipy_stats.skew(y)
    if abs(sk) > 1:
        if (y > 0).all():
            notes.append(f"Assimétrica (skew={sk:.2f}) e positiva → log/Box-Cox são candidatos (Grupo C ou D).")
        else:
            notes.append(f"Assimétrica (skew={sk:.2f}), com valores não positivos → Box-Cox precisa de shift.")

    if n_out / n > 0.05:
        notes.append(f"Muitos outliers ({n_out / n * 100:.1f}%) → RobustScaler tende a se sair melhor que Z-score/MinMax.")

    if not notes:
        notes.append("Sem padrões dominantes claros → vale testar todos os grupos.")
    for note in notes:
        print(f"  • {note}")

In [ ]:
# =========================================================
# 3. LOOP PRINCIPAL
# =========================================================

def run_preprocessing(data_dir=DATA_DIR, out_dir=OUT_DIR):
    os.makedirs(out_dir, exist_ok=True)
    files = [f for f in os.listdir(data_dir)
             if f.lower().endswith((".csv", ".xlsx", ".xls"))]
    if not files:
        print(f"Nenhum arquivo encontrado em '{data_dir}/'.")
        return {}

    saved = {}
    for f in sorted(files):
        name = os.path.splitext(f)[0]
        try:
            df = preprocess_to_univariate(os.path.join(data_dir, f))
            out_path = os.path.join(out_dir, f"{name}.csv")
            df.to_csv(out_path, index=False)
            describe_and_diagnose(df, name)
            saved[name] = out_path
        except Exception as e:
            print(f"\n[ERRO] {f}: {e}")

    print(f"\n\n{'=' * 70}")
    print(f"  ARQUIVOS PROCESSADOS SALVOS EM '{out_dir}/'")
    print('=' * 70)
    for name, path in saved.items():
        print(f"  {name:30s} → {path}")
    return saved


processed_paths = run_preprocessing()

# 2. Análise Exploratória (EDA)

Gera gráficos diagnósticos a partir dos arquivos em `data_processed/`:
série temporal, histograma + densidade, decomposição STL, ACF/PACF
e boxplots agrupados por ano e por mês.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore")

PROCESSED_DIR = "data_processed"
OUT_DIR       = "results/eda"
os.makedirs(OUT_DIR, exist_ok=True)


def detect_best_period(y, candidates=(7, 12, 24, 30, 52)):
    max_lag = min(len(y) // 2, 60)
    if max_lag < 4:
        return None
    try:
        ac = acf(y, nlags=max_lag, fft=True)
    except Exception:
        return None
    viable = [p for p in candidates if p < max_lag]
    if not viable:
        return None
    return max(viable, key=lambda p: abs(ac[p]))

In [ ]:
def plot_eda_panel(df, name):
    y = df["y"].values
    dates = pd.to_datetime(df["date"])
    n = len(y)
    period = detect_best_period(y) or 12

    fig = plt.figure(figsize=(16, 14))
    gs  = fig.add_gridspec(4, 2, hspace=0.55, wspace=0.25)

    # 1) Série temporal
    ax = fig.add_subplot(gs[0, :])
    ax.plot(dates, y, linewidth=1.1, color="steelblue")
    ax.set_title(f"{name} — Série temporal", fontsize=12, fontweight="bold")
    ax.set_xlabel("Data"); ax.set_ylabel("y")
    ax.grid(alpha=0.3)

    # 2) Histograma + densidade
    ax = fig.add_subplot(gs[1, 0])
    ax.hist(y, bins=30, color="steelblue", edgecolor="black",
            alpha=0.7, density=True)
    if np.std(y) > 0:
        kde = scipy_stats.gaussian_kde(y)
        xs = np.linspace(y.min(), y.max(), 200)
        ax.plot(xs, kde(xs), color="darkred", linewidth=2)
    sk = scipy_stats.skew(y); kt = scipy_stats.kurtosis(y)
    ax.set_title(f"Distribuição (skew={sk:.2f}, curtose={kt:.2f})", fontsize=11)
    ax.set_xlabel("y"); ax.set_ylabel("Densidade")
    ax.grid(alpha=0.3)

    # 3) Rolling mean / std (estacionariedade visual)
    ax = fig.add_subplot(gs[1, 1])
    w = max(int(n * 0.1), period)
    rm = pd.Series(y).rolling(w).mean()
    rs = pd.Series(y).rolling(w).std()
    ax.plot(dates, y, color="lightgray", linewidth=0.8, label="série")
    ax.plot(dates, rm, color="darkblue", linewidth=1.6, label=f"média móvel ({w})")
    ax.plot(dates, rs, color="darkorange", linewidth=1.6, label=f"desvio móvel ({w})")
    ax.set_title("Média e desvio móveis (estacionariedade visual)", fontsize=11)
    ax.set_xlabel("Data"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # 4) Decomposição STL
    if n >= 2 * period:
        try:
            stl = STL(y, period=period, robust=True).fit()
            ax = fig.add_subplot(gs[2, 0])
            ax.plot(dates, stl.trend, color="darkgreen", linewidth=1.2)
            ax.set_title(f"STL — Tendência (período={period})", fontsize=11)
            ax.set_xlabel("Data"); ax.grid(alpha=0.3)

            ax = fig.add_subplot(gs[2, 1])
            ax.plot(dates, stl.seasonal, color="purple", linewidth=1.0)
            ax.set_title("STL — Componente sazonal", fontsize=11)
            ax.set_xlabel("Data"); ax.grid(alpha=0.3)
        except Exception as e:
            ax = fig.add_subplot(gs[2, :])
            ax.text(0.5, 0.5, f"STL falhou: {e}",
                    ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
    else:
        ax = fig.add_subplot(gs[2, :])
        ax.text(0.5, 0.5, "Série curta demais para STL.",
                ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")

    # 5) ACF e PACF
    max_lag = min(40, n // 2 - 1)
    ax = fig.add_subplot(gs[3, 0])
    try:
        plot_acf(y, lags=max_lag, ax=ax, color="steelblue")
        ax.set_title("Autocorrelação (ACF)", fontsize=11)
    except Exception:
        ax.axis("off")

    ax = fig.add_subplot(gs[3, 1])
    try:
        plot_pacf(y, lags=max_lag, ax=ax, method="ywm", color="steelblue")
        ax.set_title("Autocorrelação parcial (PACF)", fontsize=11)
    except Exception:
        ax.axis("off")

    fig.suptitle(f"Análise exploratória — {name}",
                 fontsize=14, fontweight="bold", y=0.995)
    out = os.path.join(OUT_DIR, f"eda_{name}.png")
    fig.savefig(out, dpi=120, bbox_inches="tight")
    print(f"  salvo: {out}")
    plt.show()
    plt.close(fig)


def plot_seasonality_box(df, name):
    """
    Boxplot por ano (séries longas) ou por mês (séries com pelo menos 2 anos).
    Ajuda a ver mudanças de regime e sazonalidade anual de uma vez.
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["year"]  = df["date"].dt.year
    df["month"] = df["date"].dt.month
    n_years = df["year"].nunique()

    fig, axes = plt.subplots(1, 2 if n_years >= 2 else 1,
                             figsize=(14 if n_years >= 2 else 8, 4.5))
    if n_years < 2:
        axes = [axes]

    sns.boxplot(data=df, x="year", y="y", ax=axes[0],
                color="steelblue", fliersize=2)
    axes[0].set_title(f"{name} — Distribuição por ano")
    axes[0].set_xlabel("Ano"); axes[0].set_ylabel("y")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].grid(axis="y", alpha=0.3)

    if n_years >= 2:
        sns.boxplot(data=df, x="month", y="y", ax=axes[1],
                    color="darkorange", fliersize=2)
        axes[1].set_title(f"{name} — Distribuição por mês")
        axes[1].set_xlabel("Mês"); axes[1].set_ylabel("y")
        axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    out = os.path.join(OUT_DIR, f"sazonalidade_{name}.png")
    fig.savefig(out, dpi=120, bbox_inches="tight")
    print(f"  salvo: {out}")
    plt.show()
    plt.close(fig)

In [ ]:
files = sorted(f for f in os.listdir(PROCESSED_DIR) if f.lower().endswith(".csv"))
if not files:
    print(f"Nenhum arquivo em '{PROCESSED_DIR}/'. Rode o pré-processamento primeiro.")
else:
    print(f"Gerando gráficos em '{OUT_DIR}/'...\n")
    for f in files:
        name = os.path.splitext(f)[0]
        df = pd.read_csv(os.path.join(PROCESSED_DIR, f), parse_dates=["date"])
        plot_eda_panel(df, name)
        plot_seasonality_box(df, name)
    print("\nPronto.")

# 3. Experimento

Desenho experimental hierárquico para séries temporais com Optuna.
Modelos: regressão linear, ARIMA, XGBoost, LSTM, Transformer.
Grupos de transformação: A (só escala), B (diff sazonal), C (log/Box-Cox), D (ambos).

O test set fica intocado durante a otimização. Métricas SEMPRE na escala original.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import optuna
from scipy import stats
from scipy.special import inv_boxcox
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
from statsmodels.tsa.arima.model import ARIMA
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# N_JOBS: trials simultâneos no Optuna. Default = metade dos cores
# disponíveis. Em sistemas pequenos use 1; em servidores grandes pode
# subir mais, mas cuidado com memória (cada trial cria seu próprio modelo).
N_JOBS = max(1, (os.cpu_count() or 2) // 2)

# Quando Optuna roda em paralelo, libs internas (XGBoost/PyTorch) devem
# ser single-thread para evitar oversubscription (4 trials × 4 threads
# cada = 16 threads brigando por CPU).
_XGB_NJOBS    = 1 if N_JOBS > 1 else -1
_TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if N_JOBS > 1:
    torch.set_num_threads(1)

# Critério genérico para descartar previsões numericamente instáveis
# (independe de base): K desvios-padrão do inner_train.
SANITY_K_SIGMA = 20.0

In [ ]:
# =============================================================
# 1. CARREGAMENTO E SPLIT TEMPORAL
# =============================================================

def load_dataset(path):
    """Carrega CSV ou XLSX com 2 colunas: [data, target]."""
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)

    df = df.iloc[:, :2].copy()
    df.columns = ["date", "y"]
    df["date"] = pd.to_datetime(df["date"])
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna().sort_values("date").reset_index(drop=True)
    return df


def split_series(y, val_ratio=0.15, test_ratio=0.15):
    """Divide a série em train/val/test respeitando a ordem temporal."""
    n = len(y)
    n_test  = int(n * test_ratio)
    n_val   = int(n * val_ratio)
    n_train = n - n_val - n_test
    return (y[:n_train],
            y[n_train:n_train + n_val],
            y[n_train + n_val:])

In [ ]:
# =============================================================
# 2. MÉTRICAS (sempre na escala original)
# =============================================================

def metric_mae(yt, yp):
    return float(mean_absolute_error(yt, yp))

def metric_rmse(yt, yp):
    return float(np.sqrt(mean_squared_error(yt, yp)))

def metric_mape(yt, yp, eps=1e-8):
    return float(np.mean(np.abs((yt - yp) / (np.abs(yt) + eps))) * 100)

def metric_smape(yt, yp, eps=1e-8):
    return float(np.mean(2 * np.abs(yp - yt) /
                         (np.abs(yt) + np.abs(yp) + eps)) * 100)

def metric_mase(yt, yp, train, season=1):
    season = max(int(season), 1)
    if len(train) <= season:
        season = 1
    naive = np.mean(np.abs(train[season:] - train[:-season]))
    naive = naive if naive > 1e-8 else 1e-8
    return float(np.mean(np.abs(yt - yp)) / naive)

def all_metrics(yt, yp, train, season=1):
    return {
        "MAE":   metric_mae(yt, yp),
        "RMSE":  metric_rmse(yt, yp),
        "MAPE":  metric_mape(yt, yp),
        "sMAPE": metric_smape(yt, yp),
        "MASE":  metric_mase(yt, yp, train, season),
    }

In [ ]:
# =============================================================
# 3. PIPELINE DE TRANSFORMAÇÕES (com inversão)
# =============================================================

class TransformPipeline:
    """
    Aplica em ordem (quando configurado):
        1) Log (base 2, e ou 10) OU Box-Cox (lambda estimado no treino)
        2) Diferenciação sazonal (seasonal_period >= 1)
        3) Normalização (zscore | robust | minmax)

    A inversão segue a ordem inversa: scaler -> diff -> distribuição.
    """

    def __init__(self, cfg):
        self.cfg = dict(cfg)
        self.boxcox_lambda = None
        self.boxcox_shift  = 0.0
        self.scaler        = None
        self.pre_diff_series = None
        self._diff_applied = False

    # ---- log ----
    def _log(self, x):
        b = self.cfg.get("log_base")
        x = np.where(x <= 0, 1e-8, x)
        if b == 2:  return np.log2(x)
        if b == 10: return np.log10(x)
        return np.log(x)

    def _inv_log(self, x):
        b = self.cfg.get("log_base")
        if b == 2:  return np.power(2.0, x)
        if b == 10: return np.power(10.0, x)
        return np.exp(x)

    # ---- box-cox (lambda estimado SÓ no treino) ----
    def _fit_boxcox(self, x):
        mn = x.min()
        self.boxcox_shift = (-mn + 1.0) if mn <= 0 else 0.0
        x_pos = x + self.boxcox_shift
        x_t, lmbda = stats.boxcox(x_pos)
        self.boxcox_lambda = float(lmbda)
        return x_t

    def _inv_boxcox(self, x):
        x_back = inv_boxcox(x, self.boxcox_lambda)
        return x_back - self.boxcox_shift

    # ---- diferenciação sazonal ----
    def _inv_seasonal_diff(self, diffs, history_pre_diff):
        sp = int(self.cfg.get("seasonal_period", 0) or 0)
        hist = list(history_pre_diff)
        out  = []
        for d in diffs:
            val = d + hist[-sp]
            out.append(val)
            hist.append(val)
        return np.array(out)

    # ---- scaler ----
    def _build_scaler(self):
        s = self.cfg.get("scaler")
        if s == "zscore": return StandardScaler()
        if s == "robust": return RobustScaler()
        if s == "minmax": return MinMaxScaler()
        return None

    # ---- API pública ----
    def fit_transform(self, x):
        x = np.asarray(x, dtype=float)

        # 1) distribuição
        if self.cfg.get("log_base") is not None:
            x = self._log(x)
        elif self.cfg.get("boxcox", False):
            x = self._fit_boxcox(x)

        # guarda série pré-diff (para inverter posteriormente)
        self.pre_diff_series = x.copy()

        # 2) diferenciação sazonal — só aplica se houver dados suficientes
        sp = int(self.cfg.get("seasonal_period", 0) or 0)
        if sp > 0 and len(x) > sp:
            x = x[sp:] - x[:-sp]
            self._diff_applied = True
        else:
            self._diff_applied = False

        # 3) normalização
        self.scaler = self._build_scaler()
        if self.scaler is not None:
            x = self.scaler.fit_transform(x.reshape(-1, 1)).flatten()

        return x

    def inverse_predictions(self, y_pred):
        y = np.asarray(y_pred, dtype=float)

        if self.scaler is not None:
            y = self.scaler.inverse_transform(y.reshape(-1, 1)).flatten()
        if self._diff_applied:
            y = self._inv_seasonal_diff(y, self.pre_diff_series)
        if self.cfg.get("log_base") is not None:
            y = self._inv_log(y)
        elif self.cfg.get("boxcox", False):
            y = self._inv_boxcox(y)
        return y

In [ ]:
# =============================================================
# 4. SUGESTÕES DO OPTUNA POR GRUPO E POR MODELO
# =============================================================

def suggest_transformation_config(trial, group_name):
    cfg = {"log_base": None, "boxcox": False,
           "seasonal_period": 0, "scaler": None}

    if group_name == "A":
        cfg["scaler"] = trial.suggest_categorical(
            "scaler", ["zscore", "robust", "minmax"])

    elif group_name == "B":
        cfg["seasonal_period"] = trial.suggest_int("seasonal_period", 1, 36)
        cfg["scaler"] = trial.suggest_categorical("scaler", ["zscore", "robust"])

    elif group_name == "C":
        dist = trial.suggest_categorical("dist", ["log", "boxcox"])
        if dist == "log":
            cfg["log_base"] = trial.suggest_categorical("log_base", [2, "e", 10])
        else:
            cfg["boxcox"] = True
        cfg["scaler"] = trial.suggest_categorical("scaler", ["zscore", "robust"])

    elif group_name == "D":
        dist = trial.suggest_categorical("dist", ["log", "boxcox"])
        if dist == "log":
            cfg["log_base"] = trial.suggest_categorical("log_base", [2, "e", 10])
        else:
            cfg["boxcox"] = True
        cfg["seasonal_period"] = trial.suggest_int("seasonal_period", 1, 36)
        cfg["scaler"] = trial.suggest_categorical("scaler", ["zscore", "robust"])

    else:
        raise ValueError(f"Grupo desconhecido: {group_name}")

    return cfg


def suggest_model_params(trial, model_name):
    if model_name == "linear":
        return {"n_lags": trial.suggest_int("n_lags", 1, 24)}

    if model_name == "arima":
        return {
            "p": trial.suggest_int("p", 0, 5),
            "d": trial.suggest_int("d", 0, 2),
            "q": trial.suggest_int("q", 0, 5),
        }

    if model_name == "xgboost":
        return {
            "n_lags":        trial.suggest_int("n_lags", 1, 24),
            "n_estimators":  trial.suggest_int("n_estimators", 50, 400),
            "max_depth":     trial.suggest_int("max_depth", 2, 8),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample":     trial.suggest_float("subsample", 0.6, 1.0),
        }

    if model_name == "lstm":
        return {
            "n_lags":       trial.suggest_int("n_lags", 4, 24),
            "hidden":       trial.suggest_categorical("hidden", [16, 32, 64]),
            "layers":       trial.suggest_int("layers", 1, 2),
            "dropout":      trial.suggest_float("dropout", 0.0, 0.3),
            "lr":           trial.suggest_float("lr", 1e-3, 1e-2, log=True),
            "epochs":       trial.suggest_int("epochs", 20, 100),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        }

    if model_name == "transformer":
        return {
            "n_lags":       trial.suggest_int("n_lags", 4, 24),
            "d_model":      trial.suggest_categorical("d_model", [16, 32, 64]),
            "nhead":        trial.suggest_categorical("nhead", [2, 4]),
            "layers":       trial.suggest_int("layers", 1, 2),
            "dropout":      trial.suggest_float("dropout", 0.0, 0.3),
            "lr":           trial.suggest_float("lr", 1e-3, 1e-2, log=True),
            "epochs":       trial.suggest_int("epochs", 20, 100),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        }

    raise ValueError(f"Modelo desconhecido: {model_name}")

In [ ]:
# =============================================================
# 5. MODELOS — Arquiteturas PyTorch
# =============================================================

def make_lag_matrix(series, n_lags):
    X, y = [], []
    for i in range(n_lags, len(series)):
        X.append(series[i - n_lags:i])
        y.append(series[i])
    return np.array(X), np.array(y)


class LSTMModel(nn.Module):
    def __init__(self, hidden, layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class TransformerModel(nn.Module):
    def __init__(self, d_model, nhead, layers, dropout):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        h = self.input_proj(x)
        h = self.encoder(h)
        return self.fc(h[:, -1, :]).squeeze(-1)

In [ ]:
# =============================================================
# 5. MODELOS — Treinamento e Previsão
# =============================================================

def train_torch_model(model, X, y, lr, epochs,
                      batch_size=32, weight_decay=1e-5, patience=5):
    """
    Treina LSTM/Transformer com:
      - GPU automaticamente se disponível
      - mini-batches via DataLoader
      - early stopping em split interno (15% finais como inner-val)
      - weight_decay (regularização L2) no Adam
    """
    device = _TORCH_DEVICE
    model = model.to(device)
    n = len(X)
    opt_kwargs = dict(lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    if n < 10:
        Xt = torch.tensor(X, dtype=torch.float32, device=device).unsqueeze(-1)
        yt = torch.tensor(y, dtype=torch.float32, device=device)
        opt = torch.optim.Adam(model.parameters(), **opt_kwargs)
        model.train()
        for _ in range(epochs):
            opt.zero_grad()
            pred = model(Xt)
            loss = loss_fn(pred, yt)
            loss.backward()
            opt.step()
        return model

    # split interno cronológico para early stopping
    n_inner_val = max(2, int(n * 0.15))
    X_tr, y_tr = X[:-n_inner_val], y[:-n_inner_val]
    X_va, y_va = X[-n_inner_val:], y[-n_inner_val:]

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device).unsqueeze(-1)
    yt = torch.tensor(y_tr, dtype=torch.float32, device=device)
    Xv = torch.tensor(X_va, dtype=torch.float32, device=device).unsqueeze(-1)
    yv = torch.tensor(y_va, dtype=torch.float32, device=device)

    bs = min(batch_size, len(X_tr))
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=bs, shuffle=True)

    opt = torch.optim.Adam(model.parameters(), **opt_kwargs)

    best_val_loss = float("inf")
    best_state    = None
    no_improve    = 0

    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            v_loss = loss_fn(model(Xv), yv).item()

        if v_loss < best_val_loss - 1e-8:
            best_val_loss = v_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def fit_and_forecast(model_name, params, train_t, h):
    """
    Treina no `train_t` (já transformado) e prevê `h` passos.
    Retorna np.array(h,) na escala TRANSFORMADA.
    """
    if model_name == "arima":
        order = (params["p"], params["d"], params["q"])
        try:
            # maxiter=50: estatisticamente suficiente para a faixa p,d,q
            # do estudo (default do statsmodels é 500); ganho substancial
            # de tempo no ARIMA, que é o gargalo individual mais lento.
            mdl = ARIMA(train_t, order=order).fit(method_kwargs={"maxiter": 50})
            return np.asarray(mdl.forecast(steps=h))
        except Exception:
            return np.full(h, np.mean(train_t))

    n_lags = params["n_lags"]
    if n_lags >= len(train_t):
        return np.full(h, np.mean(train_t))

    X, y = make_lag_matrix(train_t, n_lags)
    if len(X) == 0:
        return np.full(h, np.mean(train_t))

    if model_name == "linear":
        mdl = LinearRegression().fit(X, y)
        predict_one = lambda w: float(mdl.predict(w.reshape(1, -1))[0])

    elif model_name == "xgboost":
        mdl = xgb.XGBRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            tree_method="hist",       # mais rápido em qualquer dataset
            n_jobs=_XGB_NJOBS,        # coordenado com Optuna
            random_state=RANDOM_STATE,
            verbosity=0,
        ).fit(X, y)
        predict_one = lambda w: float(mdl.predict(w.reshape(1, -1))[0])

    elif model_name == "lstm":
        mdl = LSTMModel(params["hidden"], params["layers"], params["dropout"])
        mdl = train_torch_model(mdl, X, y,
                                lr=params["lr"],
                                epochs=params["epochs"],
                                weight_decay=params["weight_decay"])
        mdl.eval()
        device = _TORCH_DEVICE
        def predict_one(w):
            with torch.no_grad():
                t = torch.tensor(w, dtype=torch.float32,
                                 device=device).reshape(1, n_lags, 1)
                return float(mdl(t).item())

    elif model_name == "transformer":
        mdl = TransformerModel(params["d_model"], params["nhead"],
                               params["layers"], params["dropout"])
        mdl = train_torch_model(mdl, X, y,
                                lr=params["lr"],
                                epochs=params["epochs"],
                                weight_decay=params["weight_decay"])
        mdl.eval()
        device = _TORCH_DEVICE
        def predict_one(w):
            with torch.no_grad():
                t = torch.tensor(w, dtype=torch.float32,
                                 device=device).reshape(1, n_lags, 1)
                return float(mdl(t).item())

    else:
        raise ValueError(model_name)

    history = list(train_t[-n_lags:])
    preds   = []
    for _ in range(h):
        window = np.array(history[-n_lags:])
        yhat   = predict_one(window)
        preds.append(yhat)
        history.append(yhat)
    return np.array(preds)

In [ ]:
# =============================================================
# 6. SANITY CHECK (genérico)
# =============================================================

def predictions_are_sane(preds, reference, k=SANITY_K_SIGMA):
    """
    Critério genérico para detectar instabilidade numérica:
    rejeita previsões que se afastam mais que k desvios-padrão da média
    do treino. Independe da base e da magnitude.

    Por que isso é metodologicamente defensável:
      - Uma série bem-comportada tem suas previsões dentro de poucos
        sigmas do passado (mesmo extremos legítimos raramente passam de
        4-5 sigmas; usamos 20 para ser conservador).
      - Configurações que produzem previsões acima de 20 sigmas
        invariavelmente sofreram falha numérica na inversão de
        transformações (ex.: log + diff sazonal acumulando erros).
      - Premiar tais configurações é estatisticamente ilegítimo.
    """
    if not np.all(np.isfinite(preds)):
        return False
    mu    = float(np.mean(reference))
    sigma = float(np.std(reference))
    if sigma <= 1e-12:
        # série constante: aceita qualquer previsão finita razoável
        return np.all(np.abs(preds - mu) < max(1.0, abs(mu)))
    return np.all(np.abs(preds - mu) < k * sigma)


# =============================================================
# 7. FUNÇÃO OBJETIVO COM VALIDAÇÃO CRUZADA TEMPORAL
# =============================================================

def objective(trial, trainval, model_name, group_name, n_splits=3):
    """
    Faz validação cruzada com janela expansiva (TimeSeriesSplit) sobre
    train+val e retorna o RMSE MÉDIO (escala original) entre dobras.

    Reporta valor parcial após cada dobra para o pruner cortar trials ruins.
    Transformações são reajustadas em CADA dobra usando apenas o inner-train.
    Configurações que produzem previsões numericamente instáveis (sanity
    check) são descartadas com inf — critério genérico baseado em sigmas
    do inner_train, não específico da base.
    """
    try:
        t_cfg    = suggest_transformation_config(trial, group_name)
        m_params = suggest_model_params(trial, model_name)

        tscv = TimeSeriesSplit(n_splits=n_splits)
        rmses = []

        for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(trainval)):
            inner_train = trainval[tr_idx]
            inner_val   = trainval[va_idx]

            pipe = TransformPipeline(t_cfg)
            train_t = pipe.fit_transform(inner_train)
            if len(train_t) < 5:
                return float("inf")

            preds_t = fit_and_forecast(model_name, m_params,
                                       train_t, h=len(inner_val))
            preds = pipe.inverse_predictions(preds_t)

            # sanity check genérico: descarta configs com instabilidade numérica
            if not predictions_are_sane(preds, inner_train):
                return float("inf")

            rmses.append(metric_rmse(inner_val, preds))

            trial.report(float(np.mean(rmses)), step=fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(rmses))

    except optuna.TrialPruned:
        raise
    except Exception:
        return float("inf")

In [ ]:
# =============================================================
# 8. RECONSTRUÇÃO DAS CONFIGURAÇÕES VENCEDORAS
# =============================================================

def build_transform_cfg_from_params(params, group_name):
    cfg = {"log_base": None, "boxcox": False,
           "seasonal_period": 0, "scaler": None}

    if group_name == "A":
        cfg["scaler"] = params.get("scaler")
    elif group_name == "B":
        cfg["seasonal_period"] = params.get("seasonal_period", 0)
        cfg["scaler"]          = params.get("scaler")
    elif group_name == "C":
        if params.get("dist") == "log":
            cfg["log_base"] = params.get("log_base")
        else:
            cfg["boxcox"] = True
        cfg["scaler"] = params.get("scaler")
    elif group_name == "D":
        if params.get("dist") == "log":
            cfg["log_base"] = params.get("log_base")
        else:
            cfg["boxcox"] = True
        cfg["seasonal_period"] = params.get("seasonal_period", 0)
        cfg["scaler"]          = params.get("scaler")

    return cfg


def build_model_params_from_params(params, model_name):
    keys_by_model = {
        "linear":         ["n_lags"],
        "arima":          ["p", "d", "q"],
        "xgboost":        ["n_lags", "n_estimators", "max_depth",
                           "learning_rate", "subsample"],
        "lstm":           ["n_lags", "hidden", "layers", "dropout",
                           "lr", "epochs", "weight_decay"],
        "transformer":    ["n_lags", "d_model", "nhead", "layers",
                           "dropout", "lr", "epochs", "weight_decay"],
    }
    keys = keys_by_model[model_name]
    return {k: params[k] for k in keys if k in params}

In [ ]:
# =============================================================
# 9. LOOP EXPERIMENTAL PRINCIPAL
# =============================================================

MODELS = ["linear", "arima", "xgboost", "lstm", "transformer"]
GROUPS = ["A", "B", "C", "D"]


def run_experiment(datasets, n_trials=50, n_splits=3, out_dir="results",
                   n_jobs=N_JOBS):
    """
    Para cada (dataset, modelo, grupo):
        - roda um estudo Optuna independente em train+val (TimeSeriesSplit)
        - reajusta a melhor config em (train+val) inteiro
        - avalia em test e armazena métricas (na escala original)

    n_jobs: número de trials simultâneos no Optuna. Default = N_JOBS
    (metade dos cores). Use 1 se tiver problemas de memória.
    """
    print(f"[config] N_JOBS Optuna={n_jobs} | XGB n_jobs={_XGB_NJOBS} | "
          f"PyTorch device={_TORCH_DEVICE}")

    os.makedirs(out_dir, exist_ok=True)
    val_rows, test_rows = [], []

    for ds_name, ds_path in datasets.items():
        df = load_dataset(ds_path)
        train, val, test = split_series(df["y"].values)
        trainval = np.concatenate([train, val])

        for model_name in MODELS:
            for group in GROUPS:
                tag = f"[{ds_name}] modelo={model_name} grupo={group}"
                print(tag)

                study = optuna.create_study(
                    direction="minimize",
                    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                    pruner=optuna.pruners.MedianPruner(
                        n_startup_trials=10, n_warmup_steps=1),
                )
                study.optimize(
                    lambda t: objective(t, trainval, model_name, group,
                                        n_splits=n_splits),
                    n_trials=n_trials,
                    n_jobs=n_jobs,
                    show_progress_bar=False,
                )

                best_params  = study.best_params
                best_cv_rmse = study.best_value
                n_pruned     = sum(1 for t in study.trials
                                   if t.state == optuna.trial.TrialState.PRUNED)

                val_rows.append({
                    "dataset":     ds_name,
                    "modelo":      model_name,
                    "grupo":       group,
                    "cv_RMSE":     best_cv_rmse,
                    "n_trials":    len(study.trials),
                    "n_pruned":    n_pruned,
                    "best_params": str(best_params),
                })

                # ---- refit em train+val e avaliação no test ----
                t_cfg    = build_transform_cfg_from_params(best_params, group)
                m_params = build_model_params_from_params(best_params, model_name)

                try:
                    pipe        = TransformPipeline(t_cfg)
                    trainval_t  = pipe.fit_transform(trainval)
                    preds_t     = fit_and_forecast(model_name, m_params,
                                                   trainval_t, h=len(test))
                    preds       = pipe.inverse_predictions(preds_t)
                    season      = max(int(t_cfg.get("seasonal_period", 1) or 1), 1)
                    metrics     = all_metrics(test, preds, trainval, season=season)
                except Exception as e:
                    metrics = {"MAE": np.nan, "RMSE": np.nan, "MAPE": np.nan,
                               "sMAPE": np.nan, "MASE": np.nan}
                    print(f"  ! falhou no refit/test: {e}")

                test_rows.append({
                    "dataset":         ds_name,
                    "modelo":          model_name,
                    "grupo":           group,
                    "transformacao":   str(t_cfg),
                    "hiperparametros": str(m_params),
                    **metrics,
                })

    val_df  = pd.DataFrame(val_rows)
    test_df = pd.DataFrame(test_rows)
    val_df.to_csv(os.path.join(out_dir, "validation_best.csv"), index=False)
    test_df.to_csv(os.path.join(out_dir, "test_results.csv"), index=False)
    print(f"\nResultados salvos em '{out_dir}/'")
    return val_df, test_df

In [ ]:
# =============================================================
# 10. EXECUÇÃO
# =============================================================

datasets = {
    # "electricity": "data_processed/electricity.csv",
}

if not datasets:
    print("Adicione caminhos em `datasets` para executar o experimento.")
else:
    val_df, test_df = run_experiment(
        datasets,
        n_trials=50,
        n_splits=3,
        out_dir="results",
        n_jobs=N_JOBS,
    )
    print("\n=== Melhor RMSE na validação cruzada ===")
    print(val_df)
    print("\n=== Métricas no teste ===")
    print(test_df)